**Import the useful packages** 

In [1]:
# Import useful packages
import numpy as np
import xarray as xr
import glob
import matplotlib.pyplot as plt
import os,sys
import pandas as pd
# To avoid problems with tensorflow, add the following lines before importing it
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow import keras
# To avoid problems with nn_functions, add the following lines before importing it 
import sys
sys.path.insert(1, '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt')
#from nn_functions.constants import *
#import nn_functions.diagnostic_functions as diag
#import nn_functions.data_formatting as dfmt
import nn_functions.postprocessing_functions as pp
#from nn_functions.constants import *
# To import various params associated with the experiment exp_name
import define_params 
# For plotting things on the polar stereographic grid with lat and lon coordinates 
import cartopy.crs as ccrs
proj=ccrs.SouthPolarStereo(central_longitude=0.0)
trans=ccrs.PlateCarree()

**Set the parameters for this specific run**

In [2]:
# Set the filepaths to useful files 
path_model = '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/NN_models/'
path_norm_metrics = '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/Training_data/'
models = glob.glob(path_model + '*.keras')
histories = glob.glob(path_model + 'history*.csv')

In [12]:
# Load in all the data
def load_test_data_whole_sim(this_collection, verbose = 0, keep_pandas = False):
    fp_apply = path_norm_metrics + this_collection + '_not_yet_normalised.csv'
    df_sel = pd.read_csv(fp_apply)
    print('\033[1m' + 'You have loaded data to apply the nn to:' + '\033[0m')
    print(fp_apply)
    print('You have kept the whole dataset')
    # Convert imported data to xarray
    if keep_pandas == False:
        clean_df = df_sel.to_xarray()
        return clean_df
    else:
        return df_sel

# Load in a trained neural network model
def load_model_nn(exp_name, this_collection, seed_nb, mod_size = 'small', TS_opt = 'extrap', norm_method = 'std', verbose = 0, 
                 annual_f = ''):
    fp_model = path_model + 'model_nn_'+ mod_size + '_' +exp_name + '_' + this_collection + '_' + annual_f + \
                                    str(seed_nb).zfill(2) + '_' + TS_opt + '_' + norm_method + '.keras'
    model = keras.models.load_model(fp_model)
    if verbose == 1:
        print('\033[1m' + 'You have loaded a trained neural network:' + '\033[0m')
        print(fp_model)
    return model

# Load in the normalisation metrics 
def load_normalisation_metrics(this_collection, norm_method = 'std', verbose = 0):
    fp_norm_metrics = path_norm_metrics + this_collection + '_metrics_norm.nc'
    norm_metrics_file = xr.open_dataset(fp_norm_metrics)
    if verbose > 0:
        print('\033[1m' + 'You have loaded normalisation metrics:' + '\033[0m')
        print(fp_norm_metrics)
    norm_metrics = norm_metrics_file.sel(norm_method=norm_method).drop('norm_method').to_dataframe()
    return norm_metrics

# Apply one trained NN to the chosen data 
def apply_model(model, norm_metrics, clean_df, exp_name, verbose = 0):
    # First normalise input data 
    val_norm = pp.normalise_vars(clean_df,
                            norm_metrics.loc['mean_vars'],
                            norm_metrics.loc['range_vars'])
    # Import the desired variables, and create a list 
    var_list = define_params.var_list_by_exp_name(exp_name)
    input_vars = list(np.array(var_list)[~(np.array(var_list) == 'melt_m_ice_per_y')])
    # Split into input and reference datasets
    x_val_norm = val_norm[input_vars]
    y_val_norm = val_norm['melt_m_ice_per_y']
    # Apply the model 
    #shape = x_val_norm.to_array().values.shape[1], x_val_norm.to_array().values.shape[0]
    #y_out_norm = model.predict(x_val_norm.to_array().values.reshape(shape),verbose = 0)
    y_out_norm = model.predict(x_val_norm.to_array().values.T,verbose = 0)
    y_out_norm_xr = xr.DataArray(data=y_out_norm.squeeze()).rename({'dim_0': 'index'})
    y_out_norm_xr = y_out_norm_xr.assign_coords({'index': x_val_norm.index})    
    # denormalise the output
    y_out = pp.denormalise_vars(y_out_norm_xr, 
                             norm_metrics['melt_m_ice_per_y'].loc['mean_vars'],
                             norm_metrics['melt_m_ice_per_y'].loc['range_vars'])
    if verbose > 0:
        print('\033[1m' + 'You have applied the neural network to the chosen data:' + '\033[0m')
    return y_out

# Apply all 10 NNs (with the 10 random seeds), and look at the mean 
def apply_nn_save_results(this_collection, exp_name, keep_all_runs = True, \
                         mod_size = 'small', TS_opt = 'extrap', norm_method = 'std', verbose = 0, annual_only = False, 
                         apply_collection = None, annual_f = ''):
    if apply_collection == None:
        load_collection = this_collection
    else:
        load_collection = apply_collection
    norm_metrics = load_normalisation_metrics(this_collection, verbose = verbose)
    clean_df = load_test_data_whole_sim(load_collection, verbose = verbose)
    if annual_only == True:
        # Get just the annual data 
        clean_pdf = clean_df.to_dataframe()
        df_annual = clean_pdf.groupby(['lat','lon','year'], as_index = False).mean()
        clean_df = df_annual.to_xarray()
    seed_nb_array = np.arange(1,11,1)
    list_melt_seeds = []
    df_ref_pred = pd.DataFrame({'lat': clean_df.lat, 
                                'lon': clean_df.lon,
                                'basin': clean_df.basins_NEMO,
                                'area': clean_df.areas, #clean_df.approx_area for OPM026?
                                'year': clean_df.year, 
                                'month': clean_df.month, 
                                'melt_m_ice_per_y': clean_df.melt_m_ice_per_y})
    for i in range(len(seed_nb_array)):
        list_melt_seeds.append('melt_' + str(i+1).zfill(2))
        if verbose == 2:
            model = load_model_nn(exp_name, this_collection, seed_nb_array[i], \
                              mod_size = mod_size, TS_opt = TS_opt, norm_method = norm_method, \
                              verbose = 1, annual_f = annual_f) 
            df_ref_pred['melt_' + str(i+1).zfill(2)] = apply_model(model, norm_metrics, clean_df, exp_name, verbose = 1)
            print(seed_nb_array[i], 'out of', len(seed_nb_array), 'processed', end = '\r')
        else:
            model = load_model_nn(exp_name, this_collection, seed_nb_array[i], \
                              mod_size = mod_size, TS_opt = TS_opt, norm_method = norm_method, \
                              verbose = 0) 
            df_ref_pred['melt_' + str(i+1).zfill(2)] = apply_model(model, norm_metrics, clean_df, exp_name, verbose = 0)
            print(seed_nb_array[i], 'out of', len(seed_nb_array), 'processed', end = '\r')
    # Calculate the ensemble mean 
    df_ref_pred['melt_pred_mean'] = np.mean(df_ref_pred[list_melt_seeds], axis = 1).values
    
    # If you want to make the files smaller, you can save the results with just the ensemble mean
    # by setting keep_all_runs to False 
    if keep_all_runs == False:
        df_ref_pred_save = df_ref_pred[list(df_ref_pred.columns.values[~np.isin(list(df_ref_pred.columns.values), list_melt_seeds)])]
        print('Warning, you have not saved the results of each nn, just the ensemble mean')
        sims = 'just_mean'
    else:
        df_ref_pred_save = df_ref_pred
        print('You have saved the results of each nn, and the ensemble mean')
        sims = 'all_sims'
    # Set the filepath and save the output
    if annual_only == False:
        ano = ''
    else:
        ano = '_annual'
    if load_collection == True:
        lc = ''
    else:
        ano = ano + '_special_lc_' + load_collection
    fp_ref_pred = path_model + 'applied_nn' + '_' + mod_size + '_' +exp_name + '_' + this_collection + '_' + annual_f + \
                                        sims + '_' + TS_opt + '_' + norm_method + ano + '.csv'
    df_ref_pred.to_csv(fp_ref_pred, index = False)
    if verbose > 0:
        print('\033[1m' + 'You have saved the results of applying the neural network to the test dataset:' + '\033[0m')
        print(fp_ref_pred)

In [10]:
this_collection = this_collection_NN
load_collection = this_collection_apply
verbose = 0


norm_metrics = load_normalisation_metrics(this_collection, verbose = verbose)
clean_df = load_test_data_whole_sim(load_collection, verbose = verbose)
print(clean_df)
if annual_only == True:
    # Get just the annual data 
    clean_pdf = clean_df.to_dataframe()
    df_annual = clean_pdf.groupby(['lat','lon','year'], as_index = False).mean()
    clean_df = df_annual.to_xarray()
seed_nb_array = np.arange(1,11,1)
list_melt_seeds = []

You have loaded data to apply the nn to:
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/Training_data/Christoph_whole_dataset_not_yet_normalised.csv
You have kept the whole dataset
<xarray.Dataset>
Dimensions:                 (index: 11838960)
Coordinates:
  * index                   (index) int64 0 1 2 3 ... 11838957 11838958 11838959
Data variables: (12/26)
    lat                     (index) float64 -85.4 -85.37 ... -64.97 -64.97
    lon                     (index) float64 -152.8 -154.1 ... -60.75 -60.5
    temperature_prop        (index) float64 34.87 34.87 34.87 ... 33.71 33.81
    salinity_prop           (index) float64 -2.017 -2.017 ... -0.647 -0.9878
    melt_m_ice_per_y        (index) float64 -3.08e-08 -8.898e-08 ... 2.458e-05
    mean_T                  (index) float64 34.87 34.87 34.87 ... 33.92 33.92
    ...                      ...
    slope_ba_lon            (index) float64 0.005232 0.008967 ... 0.0003202
    slope_ba_lat            (index) float64 0.0 0.01355 ...

NameError: name 'annual_only' is not defined

In [11]:
clean_df

<xarray.Dataset>
Dimensions:                 (index: 11838960)
Coordinates:
  * index                   (index) int64 0 1 2 3 ... 11838957 11838958 11838959
Data variables: (12/26)
    lat                     (index) float64 -85.4 -85.37 ... -64.97 -64.97
    lon                     (index) float64 -152.8 -154.1 ... -60.75 -60.5
    temperature_prop        (index) float64 34.87 34.87 34.87 ... 33.71 33.81
    salinity_prop           (index) float64 -2.017 -2.017 ... -0.647 -0.9878
    melt_m_ice_per_y        (index) float64 -3.08e-08 -8.898e-08 ... 2.458e-05
    mean_T                  (index) float64 34.87 34.87 34.87 ... 33.92 33.92
    ...                      ...
    slope_ba_lon            (index) float64 0.005232 0.008967 ... 0.0003202
    slope_ba_lat            (index) float64 0.0 0.01355 ... 0.004216 0.004216
    slope_is_across_front   (index) float64 -0.002246 0.007571 ... 0.0008544
    slope_is_towards_front  (index) float64 0.001441 0.0118 ... 0.0004521
    slope_ba_across_front   (index) float64 0.004404 0.01487 ... 0.003966
    slope_ba_towards_front  (index) float64 -0.002826 0.006565 ... -0.001463

In [ ]:
df_ref_pred = pd.DataFrame({'lat': clean_df.lat, 
                            'lon': clean_df.lon,
                            'basin': clean_df.basins_NEMO,
                            'area': clean_df.area, #clean_df.approx_area for OPM026?
                            'year': clean_df.year, 
                            'month': clean_df.month, 
                            'melt_m_ice_per_y': clean_df.melt_m_ice_per_y})
for i in range(len(seed_nb_array)):
    list_melt_seeds.append('melt_' + str(i+1).zfill(2))
    if verbose == 2:
        model = load_model_nn(exp_name, this_collection, seed_nb_array[i], \
                          mod_size = mod_size, TS_opt = TS_opt, norm_method = norm_method, \
                          verbose = 1, annual_f = annual_f) 
        df_ref_pred['melt_' + str(i+1).zfill(2)] = apply_model(model, norm_metrics, clean_df, exp_name, verbose = 1)
        print(seed_nb_array[i], 'out of', len(seed_nb_array), 'processed', end = '\r')
    else:
        model = load_model_nn(exp_name, this_collection, seed_nb_array[i], \
                          mod_size = mod_size, TS_opt = TS_opt, norm_method = norm_method, \
                          verbose = 0) 
        df_ref_pred['melt_' + str(i+1).zfill(2)] = apply_model(model, norm_metrics, clean_df, exp_name, verbose = 0)
        print(seed_nb_array[i], 'out of', len(seed_nb_array), 'processed', end = '\r')
# Calculate the ensemble mean 
df_ref_pred['melt_pred_mean'] = np.mean(df_ref_pred[list_melt_seeds], axis = 1).values

# If you want to make the files smaller, you can save the results with just the ensemble mean
# by setting keep_all_runs to False 
if keep_all_runs == False:
    df_ref_pred_save = df_ref_pred[list(df_ref_pred.columns.values[~np.isin(list(df_ref_pred.columns.values), list_melt_seeds)])]
    print('Warning, you have not saved the results of each nn, just the ensemble mean')
    sims = 'just_mean'
else:
    df_ref_pred_save = df_ref_pred
    print('You have saved the results of each nn, and the ensemble mean')
    sims = 'all_sims'
# Set the filepath and save the output
if annual_only == False:
    ano = ''
else:
    ano = '_annual'
if load_collection == True:
    lc = ''
else:
    ano = ano + '_special_lc_' + load_collection
fp_ref_pred = path_model + 'applied_nn' + '_' + mod_size + '_' +exp_name + '_' + this_collection + '_' + annual_f + \
                                    sims + '_' + TS_opt + '_' + norm_method + ano + '.csv'
df_ref_pred.to_csv(fp_ref_pred, index = False)
if verbose > 0:
    print('\033[1m' + 'You have saved the results of applying the neural network to the test dataset:' + '\033[0m')
    print(fp_ref_pred)

In [ ]:
this_collection_NN = 'OPM026_whole_dataset'
this_collection_apply = 'Christoph_whole_dataset'
exp_name = 'slope_front'

apply_nn = True
if apply_nn == True:
    apply_nn_save_results(this_collection_NN, exp_name, keep_all_runs = True, \
                         mod_size = 'small', TS_opt = 'extrap', norm_method = 'std', verbose = 2, annual_only = False, 
                         apply_collection = this_collection_apply, annual_f = '')
else:
    print('No neural network applied, change apply_nn to True if desired')

You have loaded normalisation metrics:
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/Training_data/OPM026_whole_dataset_metrics_norm.nc
You have loaded data to apply the nn to:
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/Training_data/Christoph_whole_dataset_not_yet_normalised.csv
You have kept the whole dataset
<xarray.Dataset>
Dimensions:                 (index: 11838960)
Coordinates:
  * index                   (index) int64 0 1 2 3 ... 11838957 11838958 11838959
Data variables: (12/26)
    lat                     (index) float64 -85.4 -85.37 ... -64.97 -64.97
    lon                     (index) float64 -152.8 -154.1 ... -60.75 -60.5
    temperature_prop        (index) float64 34.87 34.87 34.87 ... 33.71 33.81
    salinity_prop           (index) float64 -2.017 -2.017 ... -0.647 -0.9878
    melt_m_ice_per_y        (index) float64 -3.08e-08 -8.898e-08 ... 2.458e-05
    mean_T                  (index) float64 34.87 34.87 34.87 ... 33.92 33.92
    ...                